# LLM Unlearning — Multi-Model / Multi-Seed Sweep (free Colab T4)

Stronger experiment for the paper: **3 models x 3 seeds x 2 attacks**, reporting
mean +/- std and the outlier-vs-typical equity gap.

**Before running:** Runtime > Change runtime type > **T4 GPU**.
Expected total time on a free T4: ~1-1.5 hours. The sweep is **resumable**:
if you re-run the sweep cell, already-finished (model, seed) pairs are skipped.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi


## 2. Clone the repository


In [ ]:
!git clone https://github.com/Thommartial/llm-unlearning-experiments.git
%cd llm-unlearning-experiments


## 3. Install (PyTorch is already provided by Colab)


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/verify_gpu.py


## 4. Run the sweep (~1-1.5 h)
3 models (gpt2, gpt2-medium, pythia-410m) x 3 seeds x 2 attacks.

If you are short on time or hit disconnects, use the lighter line in the next cell
(2 models x 2 seeds). Re-running resumes from where it stopped.


In [ ]:
!python -m unlearning.sweep --config configs/sweep_tofu.yaml \
  --models gpt2 gpt2-medium EleutherAI/pythia-410m \
  --seeds 0 1 2 --out results/sweep

# Lighter fallback (faster):
# !python -m unlearning.sweep --config configs/sweep_tofu.yaml \
#   --models gpt2 gpt2-medium --seeds 0 1 --out results/sweep


## 5. Inspect the aggregated results


In [ ]:
import json
summary = json.load(open('results/sweep/sweep_summary.json'))
print(json.dumps(summary, indent=2))


In [ ]:
import pandas as pd
pd.read_csv('results/sweep/sweep_raw.csv')


In [ ]:
from IPython.display import Image
Image('results/sweep/figure_sweep.png')


## 6. Save your results
Download the sweep folder (raw CSV, summary JSON, figure) so it survives the session.


In [ ]:
!zip -r sweep_results.zip results/sweep
from google.colab import files
files.download('sweep_results.zip')


### Persisting across disconnects (optional)
Free Colab may disconnect. To resume across sessions, mount Drive and point
`--out` at a Drive folder so finished runs are remembered:
```python
from google.colab import drive; drive.mount('/content/drive')
# then: --out /content/drive/MyDrive/llm_unlearning_sweep
```

### After you have results
Send the contents of `sweep_summary.json` back, and the paper will be updated
with the multi-model / multi-seed numbers (the current paper is kept as the first draft).
